In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_2283/3320559344.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob_final") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "8")\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 15:52:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.8/dist-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or

In [ ]:
def build_final_mobile_features(spark, logical_date, logger):
    """Скрипт сборки абонентской базы для spfix_feature_store"""

    import pyspark.sql.functions as F

    logical_date = datetime.strptime(logical_date, '%Y-%m-%d').date()
    business_month = str(datetime(logical_date.year, logical_date.month, 1).date())

    logger.info('Building final dataframe...')
    logger.info(f'Logical date: {logical_date}')
    logger.info(f'Reporting date: {str(business_month)}')

    logger.info('Collecting all sources...')
    final_df = (
        spark.table('spfix_dm.mob_features_dm__1_base')
        .fillna(-1)
        .join(
            spark.table('spfix_dm.mob_features_dm__2_customer_sex_age_income'),
            ['msisdn'], 'left'
        )
        .fillna('unk')
        .join(
            spark.table('spfix_dm.mob_features_dm__3_competitors_contacts'),
            ['msisdn'], 'left'
        )
        .fillna(0)
        .repartition(700)
        .join(
            spark.table('spfix_dm.mob_features_dm__4_movements'),
            ['msisdn'], 'left'
        )
        .fillna(0)
        .repartition(700)
        .join(
            spark.table('spfix_dm.mob_features_dm__5_host_interests')
            .drop('app_n', 'regid'),
            ['msisdn'], 'left'
        )
        .fillna(0)
        .withColumn('build_dt', F.current_date())
        .withColumn('table_business_month', F.lit(business_month))
    )

    return final_df, business_month